# TRISTAR Data Preparation

This notebook prepares TRISTAR vehicle data from Gdańsk for Auto Loader testing.

In [0]:
import yaml

with open("config.yml", "r") as file:
    config = yaml.safe_load(file)

In [0]:
catalog = config["catalog"]
schema = config["schema"]
volume = config["volume"]
url = config["tristar"]["url"]

raw_path = f"/Volumes/{catalog}/{schema}/{volume}"
dataset_path  = f"{raw_path}/tristar"
staging_path = f"{dataset_path}/staging"

bronze_table = f"{catalog}.{schema}_bronze.tristar"
checkpoint_path = f"{dataset_path}/checkpoint"

## Data collection

Functions are defined to retrieve TRISTAR snapshots and save vehicle records as JSON files.

In [0]:

import json
import requests
from datetime import datetime


def get_tristar_snapshot():
    response = requests.get(url, timeout=20)
    response.raise_for_status()

    data = response.json()

    vehicles = data["vehicles"]

    timestamp = (
        datetime.fromisoformat(data["lastUpdate"].replace("Z", "+00:00"))
        .strftime("%Y%m%d_%H%M%S")
    )

    return vehicles, timestamp


def save_snapshot(vehicles, timestamp, transform=None):
    for vehicle in vehicles:
        record = vehicle.copy()

        if transform:
            record = transform(record)

        file_name = f"{record['vehicleId']}_{timestamp}.json"
        file_path = f"{staging_path}/{file_name}"

        with open(file_path, "w") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)

def add_vehicle_type(record):
    record["vehicleType"] = (
        "bus" if record["vehicleId"] % 2 == 0 else "tram"
    )

    return record


### Source data preparation

The TRISTAR API was called 3–4 times to generate approximately 1000 JSON files for ingestion testing.

In [0]:
vehicles, timestamp = get_tristar_snapshot()

save_snapshot(vehicles, timestamp)

In [0]:
len(dbutils.fs.ls(staging_path))

### Schema evolution test

A new snapshot is generated with the additional `vehicleType` column.

In [0]:
vehicles, timestamp = get_tristar_snapshot()

save_snapshot(
    vehicles,
    timestamp,
    transform=add_vehicle_type
)

print(f"Files with new schema created: {len(vehicles)}")

In [0]:
spark.table(bronze_table).printSchema()

### Rescued data test

Two records with invalid data types are created to test `_rescued_data`.

In [0]:
vehicles, timestamp = get_tristar_snapshot()

error_record_1 = vehicles[0].copy()
error_record_2 = vehicles[1].copy()

error_record_1["speed"] = "unknown"
error_record_2["lat"] = "unknown"

error_records = [error_record_1, error_record_2]

for i, record in enumerate(error_records, start=1):
    file_name = f"error_{i}_{record['vehicleId']}_{timestamp}.json"
    file_path = f"{staging_path}/{file_name}"

    with open(file_path, "w") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

print(f"Error files created: {len(error_records)}")

In [0]:
spark.table(bronze_table) \
    .select(
        "vehicleId",
        "speed",
        "lat",
        "_rescued_data",
        "source_filename"
    ) \
    .where("_rescued_data IS NOT NULL") \
    .show(truncate=False)